# Lesson 4 — Ingesting the Nigeria MTF Survey (first lesson in VS Code)

**Goal today:** extract the MTF survey data, load our first files, and make one important discovery about our target variable.

**How to work:** read each markdown cell, then run the code cell below it with **Shift+Enter**. Cells marked **YOUR TURN** are yours to complete. If anything confuses you, stop and ask your mentor — that is the system working, not failing.

## Step 1 — Imports

Remember: Python starts empty-handed. We fetch three toolboxes:
- `zipfile` — opens ZIP archives (built into Python, nothing to install)
- `pathlib.Path` — the modern way to handle file paths (also built in)
- `pandas` — our data workhorse

In [1]:
import zipfile
from pathlib import Path
import pandas as pd

print("Toolboxes loaded.")

Toolboxes loaded.


## Step 2 — Extract the raw data

We unzip *with code* rather than by right-clicking, so the notebook documents every step — anyone (including future you) can reproduce it from scratch.

Note: this notebook lives in `notebooks/`, so we reach the data with `../raw_data/` — meaning "go up one folder, then into raw_data".

In [2]:
RAW = Path("../raw_data")
EXTRACTED = RAW / "extracted"
EXTRACTED.mkdir(exist_ok=True)

with zipfile.ZipFile(RAW / "raw-data_nigeria.zip") as z:
    z.extractall(EXTRACTED)

print("Extraction complete.")

Extraction complete.


## Step 3 — See what we received

A `.dta` file is Stata's data format — very common for survey microdata from the World Bank and DHS. pandas reads it directly with `pd.read_stata()`, and it even preserves the *labels* (the human-readable answer texts), which you'll appreciate as a survey researcher.

In [3]:
dta_files = sorted(EXTRACTED.rglob("*.dta"))
for f in dta_files:
    size_kb = f.stat().st_size // 1024
    print(f"{f.name:45s} {size_kb:>8,} KB")

MTF_HH_SEC_C_BATTERY.dta                             6 KB
MTF_HH_SEC_F_FUEL.dta                              274 KB
MTF_HH_SEC_HH_ASSET_LONG.dta                       758 KB
MTF_HH_SEC_HH_ASSET_WIDE.dta                       388 KB
MTF_NG_BSS_SEC_A2.dta                            1,401 KB
MTF_NG_HH_Identification.dta                       564 KB
MTF_NG_HH_SEC_A1.dta                             2,135 KB
MTF_NG_HH_SEC_B.dta                              1,856 KB
MTF_NG_HH_SEC_C.dta                              5,480 KB
MTF_NG_HH_SEC_C_BATTERY.dta                      5,480 KB
MTF_NG_HH_SEC_C_SOLAR.dta                           27 KB
MTF_NG_HH_SEC_E.dta                              1,269 KB
MTF_NG_HH_SEC_F.dta                                550 KB
MTF_NG_HH_SEC_F_FUEL.dta                           274 KB
MTF_NG_HH_SEC_G_LAMP_CANDLE.dta                  3,908 KB
MTF_NG_HH_SEC_G_LIGHT.dta                          582 KB
MTF_NG_HH_SEC_H.dta                                402 KB
MTF_NG_HH_SEC_

Each file is one **section of the questionnaire** (open `docs/mtf_nigeria_hh_questionnaire.pdf` and you'll see the same letters: Section A = identification, B = roster, C = electricity, F = fuels...). The survey was split exactly the way you'd split a paper questionnaire.

## Step 4 — Load our first file: household identification

This file should contain one row per household with location information — including, if your instinct is right, the urban/rural variable you bet on.

In [4]:
ident = pd.read_stata(next(f for f in dta_files if "Identification" in f.name))
print("Shape (rows, columns):", ident.shape)
ident.head()

Shape (rows, columns): (3669, 14)


,HH_ID,lga,EA_code,state_id,ward,Urbanization,locality,EA_name,hhid,consent,int_start_time,language,Other_language,int_end_time
0,1.000506e+12,GUDU,100050612.0,Sokoto,BACHAKA,Rural,SALAWA,MAIGARI SAIDU,10001.0,YES,2017-11-20T06:47:00-05:00,Hausa,,2017-11-20T07:57:06-05:00
1,1.000506e+12,GUDU,100050612.0,Sokoto,BACHAKA,Rural,SALAWA,MAIGARI SAIDU,10003.0,YES,2017-11-20T06:48:43-05:00,English,,2017-11-20T07:48:29-05:00
2,1.000506e+12,GUDU,100050612.0,Sokoto,BACHAKA,Rural,SALAWA,MAIGARI SAIDU,10004.0,YES,2017-11-20T08:03:34-05:00,Hausa,,2017-11-20T09:08:46-05:00
3,1.000506e+12,GUDU,100050612.0,Sokoto,BACHAKA,Rural,SALAWA,MAIGARI SAIDU,10005.0,YES,2017-11-20T06:52:21-05:00,Hausa,,2017-11-20T07:55:20-05:00
4,1.000506e+12,GUDU,100050612.0,Sokoto,BACHAKA,Rural,SALAWA,MAIGARI SAIDU,10006.0,YES,2017-11-20T06:49:41-05:00,Hausa,,2017-11-20T07:39:52-05:00


## Step 5 — YOUR TURN: explore

Use the tools you know from your churn project. In the empty cells below, find out:
1. The column names — which column looks like urban/rural? (`ident.columns`)
2. How many households are urban vs rural? (`.value_counts()` on that column)
3. Are there duplicate households? (`ident["HH_ID"].duplicated().sum()`)

In [5]:
# YOUR TURN — column names
ident.columns

Index(['HH_ID', 'lga', 'EA_code', 'state_id', 'ward', 'Urbanization',
       'locality', 'EA_name', 'hhid', 'consent', 'int_start_time', 'language',
       'Other_language', 'int_end_time'],
      dtype='str')

In [6]:
# YOUR TURN — urban vs rural counts
ident['Urbanization'].value_counts(dropna=False)


Urbanization
Urban    1835
Rural    1833
NaN         1
Name: count, dtype: int64

In [7]:
# YOUR TURN — duplicate check
ident['HH_ID'].duplicated().sum()


np.int64(0)

## Step 6 — The solar file, and a discovery

Our project is about solar adoption, and there is a file dedicated to solar devices. Load it and look carefully at its **size**.

In [8]:
solar = pd.read_stata(next(f for f in dta_files if "SOLAR" in f.name))
print("Shape:", solar.shape)
print("Unique households owning a solar device:", solar["HH_ID"].nunique())
solar.head()

Shape: (52, 27)
Unique households owning a solar device: 43


,HH_ID,solar_id,C143,C144,C145,C146,C147,C148,C149,C150,...,C158,C158a,C159,C160,C161,C162,C163,C164,C165,C166
0,1.000506e+12,1,Module type 3W,Yes,No,Solar Home system,0.0,888,Don't Know,888,...,NaN,NaN,NaN,NaN,NaN,NaN,No,Yes,12,No problems
1,1.003514e+12,1,Saroda,Yes,No,Solar Home system,0.0,888,"Other, specify (in width and length)",888,...,NaN,NaN,NaN,NaN,NaN,NaN,No,No,6,Breaks too often
2,1.004500e+12,1,CITIZEN POWER,No,No,Solar Home system,6.0,75,75cm x150cm or larger,888,...,NaN,NaN,NaN,NaN,NaN,NaN,No,No,888,No problems
3,1.013501e+12,1,pro,Yes,No,Solar Lantern,NaN,888,Don't Know,888,...,NaN,NaN,NaN,NaN,NaN,NaN,No,Yes,6,Battery problems
4,2.011513e+12,1,Sumking,No,No,Solar Lantern,NaN,1,Don't Know,888,...,NaN,NaN,NaN,NaN,NaN,NaN,No,No,12,No problems


## Step 7 — Think before next lesson

Compare two numbers: households in the identification file (Step 4) versus unique households in the solar file (Step 6).

**Question for your mentor session:** if we define our target as "household owns a solar device," what proportion of households are adopters? Why might that be a problem for a model — and does it remind you of anything from your financial inclusion project?

There is no wrong answer here. Bring your thinking, not a polished result.

---

**Before closing:** Kernel → Restart Kernel and Run All Cells. If everything runs clean, commit:
```
git add notebooks/01_data_ingestion.ipynb
git commit -m "Lesson 4: extract MTF data, first exploration"
```